In [30]:
!pip install -q \
pinecone \
langchain \
langchain-community \
langchain-huggingface \
sentence-transformers \
pypdf

In [31]:
!pip install -U langchain langchain-core langchain-text-splitters langchain-huggingface langchain-pinecone pypdf sentence-transformers pinecone

  Using cached pinecone-9.1.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.2 kB)


In [32]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from pinecone import Pinecone, ServerlessSpec

from langchain_pinecone import PineconeVectorStore

import os

In [33]:
loader = PyPDFLoader("RAGPaper.pdf")

documents = loader.load()

In [34]:
print(len(documents))

19


In [35]:
print(documents[0].page_content)

Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric
memory have so far been only investigated for extractive downstream tas

In [36]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

In [37]:
len(docs)

183

In [38]:
print(docs[0].page_content)

Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge


In [39]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [40]:
vector = embedding_model.embed_query("What is RAG?")

print(len(vector))

384


In [41]:
PINECONE_API_KEY = "pcsk_6cBKqx_JcZxd8EGXrGaAgUJEPQCu9VWzZ1iznSGBWZrpZb7TjZS6oWPV3XT5vQpowAKNF1"

pc = Pinecone(api_key=PINECONE_API_KEY)

In [42]:
index_name = "rag-paper"

In [43]:
if index_name not in pc.list_indexes().names():

    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [44]:
vectorstore = PineconeVectorStore(
    index_name=index_name,
    embedding=embedding_model,
    pinecone_api_key=PINECONE_API_KEY
)

In [45]:
vectorstore.add_documents(docs)

['843910cc-8063-4376-9324-6e25213067ba',
 '07774e90-71b3-4fe9-84ed-715be479f127',
 'd65fc446-81ec-4b85-8c54-5241d6232ab0',
 '9e5c4cbd-1b58-403e-bd7b-1cd56d4c8c3d',
 '56eca2d4-7273-4945-a64e-4064fac82004',
 '43a0ec45-c9ce-426b-81c1-f77d84a45640',
 '0344d506-4b1b-4512-9a57-f89202d8310a',
 '552eb3a6-a921-4965-9fdc-89f7f2b8f010',
 'ed39c54a-263f-4806-b9a8-2d8589f41378',
 'ea66660d-8250-4edf-8c43-fc2252c06dad',
 '00f7920f-72c0-48d1-a056-48e4d0b13223',
 '53b8be28-62b4-4b28-bc92-4c12919c8d7c',
 '8a6a3718-b46d-407d-9cc7-2c9ccd36788e',
 '4e28bcbf-16b2-4707-8b9e-460f25667e76',
 '0bf6ac67-a358-4c81-9346-fecbe6d137a6',
 '1636a1f8-e9c5-4ad7-943e-eb4131cc3d25',
 '66e6bd0b-e01f-47e2-99df-d22416e14029',
 'e7fb52ab-b8da-4238-9a45-95b04be3dd33',
 '7a384e3c-a836-4ae5-abb9-1dd46c95b675',
 '6eacd3f6-4cdf-46d4-8975-a0e894cde947',
 '230d81a3-82d7-446d-a3e2-7d88d8e694fd',
 '5d68601d-cc45-432d-8e8c-5cc27f4ffe8f',
 '7e829155-98de-475f-b314-81fea411f7fb',
 '525363bf-7651-4721-ba8a-5476122564a6',
 '7fbcd5ca-3610-

In [46]:
query = "What is Retrieval Augmented Generation?"

results = vectorstore.similarity_search(query, k=3)

In [47]:
for i, doc in enumerate(results):
    print("="*60)
    print(i+1)
    print(doc.page_content)

1
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We conducted an thorough investigation of the learned retrieval component, validating
its effectiveness, and we illustrated how the retrieval index can be hot-swapped to update the model
2
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We conducted an thorough investigation of the learned retrieval component, validating
its effectiveness, and we illustrated how the retrieval index can be hot-swapped to update the model
3
lags behind task-speciﬁc a

In [48]:
results = vectorstore.similarity_search_with_score(
    query,
    k=5
)

for doc, score in results:

    print(score)
    print(doc.page_content[:300])

0.598613739
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We 
0.598613739
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We 
0.584068298
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric
memory have so far been only investigated for extract
0.584068298
lags behind task-speciﬁc architectures. Additiona

In [52]:
while True:

    question = input("Ask : ")

    if question.lower() == "exit":
        break

    docs = vectorstore.similarity_search(question, k=3)

    print()

    for doc in docs:
        print(doc.page_content)
        print("-"*50)

Ask : what is rag

paper, as well as HuggingFace for their help in open-sourcing code to run RAG models. The authors
would also like to thank Kyunghyun Cho and Sewon Min for productive discussions and advice. EP
thanks supports from the NSF Graduate Research Fellowship. PL is supported by the FAIR PhD
program.
References
[1] Payal Bajaj, Daniel Campos, Nick Craswell, Li Deng, Jianfeng Gao, Xiaodong Liu, Rangan
Majumder, Andrew McNamara, Bhaskar Mitra, Tri Nguyen, Mir Rosenberg, Xia Song, Alina
--------------------------------------------------
paper, as well as HuggingFace for their help in open-sourcing code to run RAG models. The authors
would also like to thank Kyunghyun Cho and Sewon Min for productive discussions and advice. EP
thanks supports from the NSF Graduate Research Fellowship. PL is supported by the FAIR PhD
program.
References
[1] Payal Bajaj, Daniel Campos, Nick Craswell, Li Deng, Jianfeng Gao, Xiaodong Liu, Rangan
Majumder, Andrew McNamara, Bhaskar Mitra, Tri Nguyen, M

In [53]:
!pip install -q -U langchain-google-genai google-generativeai

In [54]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [55]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key="AQ.Ab8RN6Ir7QE6EnCQK4Gy-6PnN3HBjp8ezmjONewycKRq4bhusg"
)

In [56]:
question = "Explain RAG."

docs = vectorstore.similarity_search(question, k=3)

context = "\n\n".join([d.page_content for d in docs])

In [57]:
prompt = f"""
Answer only using the context.

Context:

{context}

Question:

{question}
"""

In [58]:
response = llm.invoke(prompt)

print(response.content)

The provided context states that HuggingFace helped in open-sourcing code to run RAG models, and that RAG can be employed as a language model. It does not provide an explanation of what RAG is.
